# Towards Generative Modeling of CO₂ Geological Storage with Latent Conditional Diffusion Models

|| **Author: Vittoria De Pellegrini** || **Year: 2025** ||
**Institution: King Abdullah University of Science and Technology (KAUST)** ||

This notebook provides a comprehensive guide for validating a **VQ-VAE (Vector Quantized Variational Autoencoder) + GAN** Model for Pressure Build-Up Dataset.

### ⚠️ Define Normalization Strategy

To normalize our dataset to the range $[-1, 1]$, we utilize **Local Normalization**. This approach involves normalizing each sample independently based on its own minimum and maximum values. We adopt local normalization because the range of values differs significantly between samples. The formula for local normalization is calculated as follows: 

$$x_{\text{norm}} = 2 \times \frac{x - \min(x)}{\max(x) - \min(x)} - 1\tag{1}$$

Where:
- $\min(x)$ and $\max(x)$ are the minimum and maximum values of the each sample $x$ in the dataset $X = \{x_1, x_2, \dots, x_N\}$, where $N$ is the number of samples.
- $x_{\text{norm}}$ denotes the normalized value, which will falls in the range $[-1, 1]$.

Note: 
- For the Condition (porosity) we still apply **Global Normalization**, and we use it for plotting purposes.

### 📚⬇ Import Libraries and Functions 

In [ ]:
# Import libraries 
import warnings
warnings.filterwarnings('ignore')

import os 
import torch
from torch.utils.data import TensorDataset

import sys
sys.path.append('/.../')

# Import functions 
from dmfco2 import(
    set_seed,  
    dnorm_dP,
    VQVAE,   
    visualize_vqvae_reconstructed_dP_samples_val
)

### ❗️ Identify Number of Devices (Multi-GPU Training with Accelerate)

In [ ]:
# Print the number of GPUs available
num_devices = torch.cuda.device_count()
print("Number of devices: ", num_devices)

# Loop through all available devices and print their names
for i in range(num_devices):
    print(f"Device {i}: {torch.cuda.get_device_name(i)}")
    
# Print current device
print("Current device: ", torch.cuda.current_device())

### ❗️Set Seed for Reproducible Results

In [ ]:
# Set seed
set_seed(seed=42)

### 📕 Define Inputs (Conditions) and Outputs (Targets) Variables for Pressure Build-Up Validation Dataset

In [ ]:
# Define validation dataset directory path
data_dir_dP_val = '/.../co2_dataset/validation_dataset_pressure_buildup'

In [ ]:
# Define inputs (conditions) and outputs (targets) variables
val_dP_inputs = torch.load(f'{data_dir_dP_val}/dP_val_inputs.pt')
val_dP_outputs = torch.load(f'{data_dir_dP_val}/dP_val_outputs.pt')

print(val_dP_inputs.shape) # Shape: [num_samples, height, width, time_steps, num_features]
print(val_dP_outputs.shape)

In [ ]:
# Inputs -> Extract porosity (index 2) and last time frame 30.0 y (index -1)
val_dP_inputs  = val_dP_inputs[:, :, :, -1, 2]  

# Outputs -> Extract last time frame 30.0 y
val_dP_outputs = val_dP_outputs[:, :, :, -1] 

print("New shape:", val_dP_inputs.shape)
print("New shape:", val_dP_outputs.shape)

In [ ]:
# Add number of features (1) to inputs (conditions) and outputs (targets)v
val_dP_inputs = val_dP_inputs[:, :, :].unsqueeze(1) # Shape will be [4500, 1, 96, 200]
val_dP_outputs = val_dP_outputs[:, :, :].unsqueeze(1) # Shape will be [4500, 1, 96, 200]

# Print new shape
print("New shape:", val_dP_inputs.shape)   # Expected: [num_samples, num_features, height, width] -> Only Porosity as Condition 
print("New shape:", val_dP_outputs.shape)  # Expected: [num_samples, num_features, height, width] -> Only Pressure Build-Up @ year 30.0

In [ ]:
# Apply dnorm_dP function to ensure proper Pressure Build-Up range
val_dP_outputs=dnorm_dP(val_dP_outputs)

# Check Min and Max values 
min_val_dP = val_dP_outputs.min()
max_val_dP = val_dP_outputs.max()
print(min_val_dP) 
print(max_val_dP)

In [ ]:
# Check ranges for each sample individually (CONDITION)
# Reduce over dimensions 1, 2, 3 to compute min and max for each sample
min_val_por_individual = val_dP_inputs.amin(dim=(1, 2, 3))  # Min per sample
max_val_por_individual = val_dP_inputs.amax(dim=(1, 2, 3))  # Max per sample

# Verify shapes
print(f"Shape of min_val_por_individual: {min_val_por_individual.shape}") # [500]
print(f"Shape of max_val_por_individual: {max_val_por_individual.shape}") # [500]

# Combine min and max into ranges
ranges = [(min_val.item(), max_val.item()) for min_val, max_val in zip(min_val_por_individual, max_val_por_individual)]

# Display all the ranges
for i, r in enumerate(ranges[:]): 
    print(f"Sample {i + 1} range: {r}")

In [ ]:
# Normalize inputs between -1 and 1 
min_val_por = val_dP_inputs.min()
max_val_por = val_dP_inputs.max()

# Check min and max values in val_dP_inputs before global normalization
print("Global Min for Porosity before normalization:", val_dP_inputs.min().item(), "Global Max for Porosity before normalization:", val_dP_inputs.max().item())

# Global Normalization formula
val_dP_inputs = 2 * (val_dP_inputs - min_val_por) / (max_val_por - min_val_por) - 1

# Check min and max values in val_dP_inputs after global normalization
print("Global Min for Porosity after normalization:", val_dP_inputs.min().item(), "Global Min for Porosity after normalization:", val_dP_inputs.max().item())

In [ ]:
# Some dP values are negative. Replace negative values with zeros
val_dP_outputs = torch.clamp(val_dP_outputs, min=0)

In [ ]:
# Check ranges for each sample individually (TARGET)
# Reduce over dimensions 1, 2, 3 to compute min and max for each sample
min_val_dP_individual = val_dP_outputs.amin(dim=(1, 2, 3))  # Min per sample
max_val_dP_individual = val_dP_outputs.amax(dim=(1, 2, 3))  # Max per sample

# Verify shapes
print(f"Shape of min_val_dP_individual: {min_val_dP_individual.shape}")  # [500]
print(f"Shape of max_val_dP_individual: {max_val_dP_individual.shape}")  # [500]

# Combine min and max into ranges
ranges = [(min_val.item(), max_val.item()) for min_val, max_val in zip(min_val_dP_individual, max_val_dP_individual)]

# Display all the ranges
for i, r in enumerate(ranges[:]): 
    print(f"Sample {i + 1} range: {r}")
    
# Check min and max values in val_dP_outputs before local normalization
print("Global Min for Pressure Build-Up before normalization:", val_dP_outputs.min().item(), "Global Max for Pressure Build-Up before normalization:", val_dP_outputs.max().item())

In [ ]:
# Ensure logging directory exists
log_dir = "/.../co2_dataset/log_files/"
os.makedirs(log_dir, exist_ok=True)
dP_log_file = os.path.join(log_dir, "val_dP_min_max_before_normalization.txt")

# Save min and max values before normalization into a text file
with open(dP_log_file, "w") as log_file:
    log_file.write("Sample-wise Min-Max Values Before Normalization\n")
    log_file.write("=" * 80 + "\n")
    for i in range(len(min_val_dP_individual)):
        log_file.write(f"Sample {i+1}: Min Before={min_val_dP_individual[i].item():.4f}, Max Before={max_val_dP_individual[i].item():.4f}\n")

In [ ]:
# Normalize outputs between -1 and 1 

# Initialize arrays to store min and max values for each sample
min_val_dP = []
max_val_dP = []

# Normalize outputs and store min/max values
for i in range(len(val_dP_outputs)):
    min_val = val_dP_outputs[i].min()
    max_val = val_dP_outputs[i].max()
    print(f"Sample {i + 1}:")
    print(f"Original Min: {min_val}, Original Max: {max_val}")
    min_val_dP.append(min_val)
    max_val_dP.append(max_val)

    # Local Normalization formula
    val_dP_outputs[i] = 2 * (val_dP_outputs[i] - min_val) / (max_val - min_val) - 1
    print(f"Normalized Output: {val_dP_outputs[i]}")

In [ ]:
# Check ranges for each sample individually (TARGET)
# Reduce over dimensions 1, 2, 3 to compute min and max for each sample
min_val_dP_individual_after_norm = val_dP_outputs.amin(dim=(1, 2, 3))  # Min per sample
max_val_dP_individual_after_norm = val_dP_outputs.amax(dim=(1, 2, 3))  # Max per sample

# Verify shapes
print(f"Shape of min_val_dP_individual_after_norm: {min_val_dP_individual_after_norm.shape}")  # [500]
print(f"Shape of max_val_dP_individual_after_norm: {max_val_dP_individual_after_norm.shape}")  # [500]

# Combine min and max into ranges
ranges = [(min_val.item(), max_val.item()) for min_val, max_val in zip(min_val_dP_individual_after_norm, max_val_dP_individual_after_norm)]

# Display all the ranges
for i, r in enumerate(ranges[:]): 
    print(f"Sample {i + 1} range: {r}")
    
# Check min and max values in val_dP_outputs after local normalization
print("Global Min for Pressure Build-Up after normalization:", val_dP_outputs.min().item(), "Global Min for Pressure Build-Up after normalization:", val_dP_outputs.max().item())

In [ ]:
# Ensure logging directory exists
log_dir = "/.../co2_dataset/log_files/"
os.makedirs(log_dir, exist_ok=True)
dP_log_file = os.path.join(log_dir, "val_dP_min_max_after_normalization.txt")

# Save min and max values before normalization into a text file
with open(dP_log_file, "w") as log_file:
    log_file.write("Sample-wise Min-Max Values After Normalization\n")
    log_file.write("=" * 80 + "\n")
    for i in range(len(min_val_dP_individual_after_norm)):
        log_file.write(f"Sample {i+1}: Min After={min_val_dP_individual_after_norm[i].item():.4f}, Max After={max_val_dP_individual_after_norm[i].item():.4f}\n")

In [ ]:
# Define variables for conditional diffusion model
val_dP_label = val_dP_inputs
val_dP_target = val_dP_outputs

### 🗂️ Define Validation Dataset

In [ ]:
# Create dataloaders
validation_dataset = TensorDataset(val_dP_target, val_dP_label)

### 🚀 Load Pre-Trained VQ-VAE + GAN Model

In [ ]:
# Load saved model
vqvae_checkpoint = '/.../trained_models/pressure_buildup_models/vqvae/best_model/vqvae_model.pth' 
vqvae_checkpoint = torch.load(vqvae_checkpoint)

# Extract parameters and weights
z_channels = vqvae_checkpoint["z_channels"]
codebook_size = vqvae_checkpoint["codebook_size"]
state_dict = vqvae_checkpoint["state_dict"]

# Initialize the model with the saved parameters
vqvae_model = VQVAE(z_channels=z_channels, codebook_size=codebook_size)

# Load the model weights
vqvae_model.load_state_dict(state_dict)

print("Model loaded successfully with z_channels={} and codebook_size={}".format(z_channels, codebook_size))

### 🔗 Save Reconstructed Samples

In [ ]:
# Define save directory for loss curves
save_dir = "/.../results/vqvae_pressure_buildup_results/reconstructed_samples_validation_dataset"

# Specify the first 500 sample indices
sample_indices = list(range(500))

# Call the function for Pressure Build-Up
visualize_vqvae_reconstructed_dP_samples_val(
    model=vqvae_model,
    dataset=validation_dataset,
    sample_indices=sample_indices,
    min_val_dP=min_val_dP,
    max_val_dP=max_val_dP,
    min_val_por=min_val_por,
    max_val_por=max_val_por,
    save_dir=save_dir
)

## References 

**[1] Wen et al., 2002** ([Paper](https://www.sciencedirect.com/science/article/pii/S0309170822000562)): CO₂ Open-Source Dataset ([Google Drive Link](https://drive.google.com/drive/folders/1fZQfMn_vsjKUXAfRV0q_gswtl8JEkVGo)).